# Défi : Analyse de texte sur la science des données

Dans cet exemple, faisons un exercice simple qui couvre toutes les étapes d'un processus traditionnel de science des données. Vous n'avez pas à écrire de code, vous pouvez simplement cliquer sur les cellules ci-dessous pour les exécuter et observer le résultat. Comme défi, nous vous encourageons à essayer ce code avec différentes données.

## Objectif

Dans cette leçon, nous avons discuté de différents concepts liés à la science des données. Essayons de découvrir plus de concepts connexes en faisant un **extraction de texte**. Nous commencerons par un texte sur la science des données, en extrayant des mots-clés, puis en essayant de visualiser le résultat.

Pour le texte, j'utiliserai la page sur la science des données de Wikipédia :


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Étape 1 : Récupération des données

La première étape de tout processus de science des données est la récupération des données. Nous allons utiliser la bibliothèque `requests` pour cela :


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Étape 2 : Transformer les données

L'étape suivante consiste à convertir les données sous une forme adaptée au traitement. Dans notre cas, nous avons téléchargé le code source HTML de la page, et nous devons le convertir en texte brut.

Il existe de nombreuses façons de le faire. Nous allons utiliser [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), une bibliothèque Python populaire pour analyser le HTML. BeautifulSoup nous permet de cibler des éléments HTML spécifiques, afin que nous puissions nous concentrer sur le contenu principal de l'article de Wikipedia et réduire certains menus de navigation, barres latérales, pieds de page, et autres contenus non pertinents (bien que du texte standard puisse encore rester).


Tout d'abord, nous devons installer la bibliothèque BeautifulSoup pour l'analyse HTML :


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Étape 3 : Obtenir des insights

L'étape la plus importante est de transformer nos données en une forme à partir de laquelle nous pouvons tirer des insights. Dans notre cas, nous voulons extraire des mots-clés du texte et voir quels mots-clés sont les plus significatifs.

Nous allons utiliser une bibliothèque Python appelée [RAKE](https://github.com/aneesha/RAKE) pour l'extraction des mots-clés. Tout d'abord, installons cette bibliothèque au cas où elle ne serait pas présente : 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

La fonctionnalité principale est disponible à partir de l'objet `Rake`, que nous pouvons personnaliser en utilisant certains paramètres. Dans notre cas, nous allons définir la longueur minimale d'un mot-clé à 5 caractères, la fréquence minimale d'un mot-clé dans le document à 3, et le nombre maximal de mots dans un mot-clé - à 2. N'hésitez pas à expérimenter avec d'autres valeurs et à observer le résultat.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Nous avons obtenu une liste de termes avec le degré d'importance associé. Comme vous pouvez le voir, les disciplines les plus pertinentes, telles que l'apprentissage automatique et le big data, sont présentes en haut de la liste.

## Étape 4 : Visualisation du résultat

Les gens peuvent mieux interpréter les données sous forme visuelle. Il est donc souvent judicieux de visualiser les données afin d'en tirer des enseignements. Nous pouvons utiliser la bibliothèque `matplotlib` en Python pour tracer la distribution simple des mots-clés avec leur pertinence :


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Il existe cependant une manière encore meilleure de visualiser les fréquences des mots : en utilisant un **Word Cloud**. Nous aurons besoin d’installer une autre bibliothèque pour tracer le nuage de mots à partir de notre liste de mots-clés.


In [ ]:
!{sys.executable} -m pip install wordcloud

L'objet `WordCloud` est responsable de la prise en charge soit du texte original, soit d'une liste pré-calculée de mots avec leurs fréquences, et renvoie une image, qui peut ensuite être affichée à l'aide de `matplotlib` :


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Nous pouvons également passer le texte original à `WordCloud` - voyons si nous sommes capables d’obtenir un résultat similaire :


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Vous pouvez voir que le nuage de mots semble maintenant plus impressionnant, mais il contient aussi beaucoup de bruit (par exemple des mots sans rapport comme `Retrieved on`). De plus, nous obtenons moins de mots-clés composés de deux mots, tels que *data scientist* ou *computer science*. Cela s'explique par le fait que l'algorithme RAKE fait un bien meilleur travail pour sélectionner de bons mots-clés à partir du texte. Cet exemple illustre l'importance du pré-traitement et du nettoyage des données, car une image claire à la fin nous permettra de prendre de meilleures décisions.

Dans cet exercice, nous avons suivi un processus simple d'extraction de sens à partir du texte Wikipedia, sous forme de mots-clés et de nuage de mots. Cet exemple est assez simple, mais il montre bien toutes les étapes typiques qu'un data scientist suivra lorsqu'il travaille avec des données, depuis l'acquisition des données jusqu'à la visualisation.

Dans notre cours, nous discuterons tous ces étapes en détail.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
